<a href="https://colab.research.google.com/github/snumryk/TRPA1-ML-benchmark/blob/main/scripts/DatasetHandler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Встановлення необхідних бібліотек
# Встановлення клієнта ChEMBL, PubChemPy та RDKit для обробки хімічних структур
!pip install chembl_webresource_client pubchempy rdkit pandas requests tqdm seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.7 MB/s eta 0:00:00


In [ ]:
# @title 2. Повний код генерації та курації датасету TRPA1
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import SaltRemover
from tqdm.auto import tqdm
import requests
import io
import re

# Налаштування відображення pandas
pd.set_option('display.max_columns', None)

# --- БЛОК 1: Функції для роботи з ChEMBL ---

def classify_mechanism_chembl(row):
    """
    Класифікує сполуку як Агоніст або Антагоніст на основі опису аналізу (assay_description)
    та типу стандартної активності. Використовує евристичний пошук ключових слів.
    """
    desc = str(row.get('assay_description', '')).lower()
    std_type = str(row.get('standard_type', '')).upper()

    # Ключові слова для Антагоністів (інгібіторів)
    antagonist_keywords = ['inhibit', 'antagonist', 'block', 'reduction', 'suppress', 'decrease']
    # Ключові слова для Агоністів (активаторів)
    agonist_keywords = ['agonist', 'activat', 'induce', 'stimulat', 'increase', 'current']

    # Логіка визначення
    is_antagonist = any(k in desc for k in antagonist_keywords)
    is_agonist = any(k in desc for k in agonist_keywords)

    # Вирішення конфліктів та пріоритезація
    if is_antagonist and not is_agonist:
        return 'Antagonist'
    elif is_agonist and not is_antagonist:
        return 'Agonist'
    elif is_antagonist and is_agonist:
        # Якщо є обидва типи слів, дивимось на тип метрики
        if std_type in ['IC50', 'KI', 'INH']:
            return 'Antagonist'
        elif std_type in ['EC50', 'AC50']:
            return 'Agonist'
        else:
            return 'Ambiguous' # Потребує ручної перевірки
    else:
        # Якщо опис неінформативний, покладаємось виключно на метрику
        if std_type in ['IC50', 'KI']:
            return 'Antagonist'
        if std_type == 'EC50':
            return 'Agonist'

    return 'Unclassified'

def fetch_chembl_data(target_chembl_id="CHEMBL6007"):
    """
    Завантажує дані біоактивності для вказаної мішені з ChEMBL.
    Фільтрує за організмом (Homo sapiens) та типами активності.
    """
    print(f"--- Початок завантаження даних з ChEMBL для {target_chembl_id} ---")
    activity = new_client.activity

    # Фільтрація на рівні API для зменшення обсягу переданих даних
    res = activity.filter(
        target_chembl_id=target_chembl_id,
        target_organism="Homo sapiens",
        standard_type__in=["IC50", "EC50", "Ki", "Kd"],
        relation="=" # Тільки точні значення
    ).only(
        "molecule_chembl_id", "canonical_smiles", "standard_type",
        "standard_value", "standard_units", "assay_description", "type", "assay_type"
    )

    df = pd.DataFrame.from_dict(res)
    print(f"Завантажено {len(df)} сирих записів з ChEMBL.")

    # Попередня обробка
    # Видалення записів без SMILES або значень
    df = df.dropna(subset=['canonical_smiles', 'standard_value'])

    # Конвертація значень у числовий формат
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')

    # Стандартизація одиниць вимірювання (все в nM)
    # Якщо одиниці uM, множимо на 1000, якщо nM - залишаємо.
    # Для спрощення припускаємо, що ChEMBL standard_value зазвичай вже нормалізовані,
    # але фільтруємо для певності.
    df = df[df['standard_units'] == 'nM']

    # Застосування класифікації
    df['mode_of_action'] = df.apply(classify_mechanism_chembl, axis=1)

    # Фільтрація некласифікованих
    df_classified = df[df['mode_of_action'].isin(['Agonist', 'Antagonist'])]
    print(f"Після класифікації та фільтрації залишилось {len(df_classified)} записів.")

    # Додавання джерела
    df_classified['source_db'] = 'ChEMBL'

    return df_classified

# --- БЛОК 2: Функції для роботи з BindingDB ---

def fetch_bindingdb_data(uniprot_id="O75762"):
    """
    Завантажує дані з BindingDB через REST API за UniProt ID.
    """
    print(f"\n--- Початок завантаження даних з BindingDB для {uniprot_id} ---")
    url = f"https://www.bindingdb.org/axis2/services/BDBService/getLigandsByUniprot?uniprot={uniprot_id}&response=json"

    try:
        response = requests.get(url)
        if response.status_code == 200:
            # Парсинг відповіді (BindingDB повертає специфічний JSON/XML)
            # Примітка: Реальна структура відповіді BindingDB може бути складною.
            # Тут реалізовано базову обробку. Для великих даних краще використовувати TSV дампи.
            try:
                data = response.json()
                # Трансформація JSON у DataFrame
                # Структура BindingDB JSON: ligands -> list of dicts
                if 'affinities' in data: # Приклад ключа, може відрізнятись
                     pass # Потрібен детальний парсинг залежно від актуальної схеми

                # Оскільки API BindingDB часто змінюється або повертає XML,
                # надійнішим методом є завантаження TSV, якщо API не відповідає очікуванням.
                # Для демонстрації, припустимо, що ми отримали дані.
                print("З'єднання з BindingDB встановлено. (Для повної реалізації потрібен парсинг XML/JSON структури)")
                return pd.DataFrame() # Повертаємо пустий DF як заглушку, якщо парсинг складний в межах скрипта
            except:
                print("Не вдалося розпарсити JSON BindingDB (можливо повернувся XML або пустий результат).")
                return pd.DataFrame()
        else:
            print(f"Помилка запиту до BindingDB: {response.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Виключення при роботі з BindingDB: {e}")
        return pd.DataFrame()

# --- БЛОК 3: Функції стандартизації хімічних структур (RDKit) ---

def standardize_structure(smiles):
    """
    Стандартизує SMILES:
    1. Канонізація.
    2. Видалення солей (Salt Stripping).
    3. Перевірка валідності.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Видалення солей
        remover = SaltRemover.SaltRemover()
        mol = remover.StripMol(mol)

        # Повернення канонічного ізомерного SMILES
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except:
        return None

# --- БЛОК 4: Головний конвеєр обробки ---

def run_pipeline():
    # 1. Завантаження даних з ChEMBL
    df_chembl = fetch_chembl_data()

    # 2. Стандартизація структур (це може зайняти час)
    print("\n--- Стандартизація хімічних структур (RDKit) ---")
    tqdm.pandas()
    df_chembl['std_smiles'] = df_chembl['canonical_smiles'].progress_apply(standardize_structure)

    # Видалення записів, де стандартизація не вдалася
    df_clean = df_chembl.dropna(subset=['std_smiles'])

    # 3. Обчислення pChEMBL (pIC50/pEC50)
    # pValue = -log10(Molar concentration)
    # standard_value в nM, тому: value * 1e-9 -> M
    df_clean['p_value'] = -np.log10(df_clean['standard_value'] * 1e-9)

    # 4. Агрегація дублікатів
    # Якщо одна сполука має кілька вимірювань, беремо середнє геометричне (середнє арифметичне p-значень)
    print("Агрегація дублікатів...")
    df_agg = df_clean.groupby(['std_smiles', 'mode_of_action']).agg({
        'p_value': 'mean',
        'molecule_chembl_id': 'first', # Беремо перший ID як представника
        'source_db': 'first'
    }).reset_index()

    # 5. Фінальний аналіз
    print("\n--- Підсумкова статистика датасету ---")
    print(df_agg['mode_of_action'].value_counts())

    # Збереження
    df_agg.to_csv("TRPA1_Curated_Dataset.csv", index=False)
    print("Датасет збережено у файл 'TRPA1_Curated_Dataset.csv'")

    return df_agg

# Запуск пайплайну (розкоментуйте для запуску в Colab)
# final_dataset = run_pipeline()